In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
import plotly.graph_objects as go

In [4]:
evasao_reprovacao = pd.read_csv('https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/Data_Cleaning/src/Datas/Maiores_Taxas_Evasao_e_Reprovacao_2024.csv')
# codigo_escola

evasao_aproveitamento = pd.read_csv('/content/menores_taxas_evasao_e_reprovacao_2024.csv', sep=';')

menores_taxas = pd.read_csv('/content/menores_taxas.csv', sep=';')

censo_escolar = pd.read_csv('https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/Data_Cleaning/src/Datas/Tabela_censo_escolar_2024.csv', sep=';')
# CO_ENTIDADE

grafico_escolas = pd.read_csv('https://raw.githubusercontent.com/MarcilioFilh0/Data_Analysis_Education/refs/heads/Data_Table_Relationships/src/Datas/censo_filtrado.csv', sep=';')

In [5]:
codigos_filtrados = evasao_reprovacao['codigo_escola'].unique()

censo_filtrado = censo_escolar[censo_escolar['CO_ENTIDADE'].isin(codigos_filtrados)]

# censo_filtrado.to_csv('censo_filtrado.csv', sep=';', index=False)

In [6]:
tabela_nova = evasao_aproveitamento['codigo_escola'].unique()

tabela_novas = censo_escolar[censo_escolar['CO_ENTIDADE'].isin(tabela_nova)]

# tabela_novas.to_csv('menores_taxas.csv', sep=';', index=False)

In [7]:
colunas_infra = [
    'IN_AGUA_POTAVEL', 'IN_AGUA_INEXISTENTE', 'IN_ENERGIA_REDE_PUBLICA',
    'IN_ENERGIA_INEXISTENTE', 'IN_ESGOTO_REDE_PUBLICA', 'IN_ESGOTO_INEXISTENTE',
    'IN_BANHEIRO', 'IN_COZINHA', 'IN_LABORATORIO_CIENCIAS', 'IN_LABORATORIO_INFORMATICA',
    'IN_QUADRA_ESPORTES', 'IN_REFEITORIO', 'IN_SALA_MULTIUSO', 'IN_SALA_DIRETORIA',
    'IN_SALA_LEITURA', 'IN_SECRETARIA', 'IN_INTERNET', 'IN_INTERNET_ALUNOS', 'IN_ALIMENTACAO'
]

def criar_tabela_dinamica(coluna_selecionada):
    dados_filtrados = grafico_escolas[grafico_escolas[coluna_selecionada].isin([0.0, 1.0])]

    tabela_dinamica = pd.pivot_table(
        dados_filtrados,
        values='CO_ENTIDADE',       # Conta a quantidade de escolas
        index='SG_UF',              # Linhas da tabela (Estados)
        columns=coluna_selecionada, # Colunas da tabela (0.0 ou 1.0)
        aggfunc='count',            # Função de agregação: contagem
        fill_value=0                # Substitui estados sem dados por 0
    )

    tabela_dinamica.columns = ['Não possui', 'Possui']
    tabela_dinamica['Total Geral'] = tabela_dinamica['Não possui'] + tabela_dinamica['Possui']
    return tabela_dinamica

interact(criar_tabela_dinamica, coluna_selecionada=colunas_infra);

interactive(children=(Dropdown(description='coluna_selecionada', options=('IN_AGUA_POTAVEL', 'IN_AGUA_INEXISTE…

In [8]:
colunas_analise = [    'IN_AGUA_POTAVEL', 'IN_AGUA_INEXISTENTE', 'IN_ENERGIA_REDE_PUBLICA',
    'IN_ENERGIA_INEXISTENTE', 'IN_ESGOTO_REDE_PUBLICA', 'IN_ESGOTO_INEXISTENTE',
    'IN_BANHEIRO', 'IN_COZINHA', 'IN_LABORATORIO_CIENCIAS', 'IN_LABORATORIO_INFORMATICA',
    'IN_QUADRA_ESPORTES', 'IN_REFEITORIO', 'IN_SALA_MULTIUSO', 'IN_SALA_DIRETORIA',
    'IN_SALA_LEITURA', 'IN_SECRETARIA', 'IN_INTERNET', 'IN_INTERNET_ALUNOS', 'IN_ALIMENTACAO']

df_agrupado = menores_taxas.groupby('SG_UF')[colunas_analise].sum().reset_index()
fig = go.Figure()

for i, coluna in enumerate(colunas_analise):
    fig.add_trace(
        go.Bar(
            x=df_agrupado['SG_UF'],
            y=df_agrupado[coluna],
            name=coluna.replace('IN_', '').replace('_', ' '),
            visible=(i == 0),
            marker_color='#111',
        )
    )

botoes = []
for i, coluna in enumerate(colunas_analise):
    visibilidade = [False] * len(colunas_analise)
    visibilidade[i] = True

    texto_botao = coluna.replace('IN_', '').replace('_', ' ').title()

    botoes.append(
        dict(
            label=texto_botao,
            method='update',
            args=[
                {'visible': visibilidade},
                {
                    'title': f'Quantidade de Escolas por Estado: {texto_botao}'
                },
            ],
        )
    )

fig.update_layout(
    title={
        'text': f"Quantidade de Escolas por Estado: {colunas_analise[0].replace('IN_', '').replace('_', ' ').title()}",
        'y': 0.89,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title='Estado (UF)',
    yaxis_title='Quantidade de Escolas',
    template='plotly_white',
    margin=dict(t=150),
    updatemenus=[
        dict(
            active=0,
            buttons=botoes,
            direction='down',
            pad={'r': 10, 't': 10},
            showactive=True,
            x=0.5,
            xanchor='center',
            y=1.25,
            yanchor='top',
        )
    ],
)

fig.show()

In [9]:
df_agrupado = censo_escolar.groupby('SG_UF')[colunas_infra].sum().reset_index()
fig = go.Figure()

colunas_analise = [    'IN_AGUA_POTAVEL', 'IN_AGUA_INEXISTENTE', 'IN_ENERGIA_REDE_PUBLICA',
    'IN_ENERGIA_INEXISTENTE', 'IN_ESGOTO_REDE_PUBLICA', 'IN_ESGOTO_INEXISTENTE',
    'IN_BANHEIRO', 'IN_COZINHA', 'IN_LABORATORIO_CIENCIAS', 'IN_LABORATORIO_INFORMATICA',
    'IN_QUADRA_ESPORTES', 'IN_REFEITORIO', 'IN_SALA_MULTIUSO', 'IN_SALA_DIRETORIA',
    'IN_SALA_LEITURA', 'IN_SECRETARIA', 'IN_INTERNET', 'IN_INTERNET_ALUNOS', 'IN_ALIMENTACAO']

for i, coluna in enumerate(colunas_analise):
    fig.add_trace(
        go.Bar(
            x=df_agrupado['SG_UF'],
            y=df_agrupado[coluna],
            name=coluna.replace('IN_', '').replace('_', ' '),
            visible=(i == 0),
            marker_color='#111',
        )
    )

botoes = []
for i, coluna in enumerate(colunas_analise):
    visibilidade = [False] * len(colunas_analise)
    visibilidade[i] = True

    texto_botao = coluna.replace('IN_', '').replace('_', ' ').title()

    botoes.append(
        dict(
            label=texto_botao,
            method='update',
            args=[
                {'visible': visibilidade},
                {
                    'title': f'Quantidade de Escolas por Estado: {texto_botao}'
                },
            ],
        )
    )

fig.update_layout(
    title={
        'text': f"Quantidade de Escolas por Estado: {colunas_analise[0].replace('IN_', '').replace('_', ' ').title()}",
        'y': 0.89,
        'x': 0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    xaxis_title='Estado (UF)',
    yaxis_title='Quantidade de Escolas',
    template='plotly_white',
    margin=dict(t=150),
    updatemenus=[
        dict(
            active=0,
            buttons=botoes,
            direction='down',
            pad={'r': 10, 't': 10},
            showactive=True,
            x=0.5,
            xanchor='center',
            y=1.25,
            yanchor='top',
        )
    ],
)

fig.show()